# F02 OCULUS — PALATIUM
> *Frégate Viewer — Configuration Caméra, Éclairage, Tags*

**Pipeline :**
1. Monter Drive
2. Valider les fichiers IN/ avec L'Arbitre
3. Démarrer Flask + ngrok
4. Accéder au viewer via l'URL ngrok
5. Configurer spawn, waypoints, éclairage, tags
6. Sauvegarder (Ctrl+S dans le viewer)
7. Valider OUT/ avec L'Arbitre

---
**Doctrine :** F02 ne lit que `F02_OCULUS/IN/` et n'écrit que dans `F02_OCULUS/OUT/`.

In [ ]:
# ─── CONFIGURATION ─────────────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/DRIVE_PALATIUM'
FLASK_PORT  = 5002
# ───────────────────────────────────────────────────────────

## Étape 1 — Monter Drive + Installer dépendances

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install flask flask-cors -q
print('✅ Setup complet')

## Étape 1b — Transit SHARED → F02_OCULUS/IN/

> Copie `maison.glb` et les fichiers HDRI depuis `SHARED/` vers `F02_OCULUS/IN/`.
> Cellule idempotente — sans danger de relancer si les fichiers sont déjà présents.

In [ ]:
import shutil, os

SHARED = DRIVE_ROOT + '/SHARED'
IN     = DRIVE_ROOT + '/F02_OCULUS/IN'

shutil.copy(SHARED + '/maison.glb', IN + '/maison.glb')
print('✅ maison.glb → F02_OCULUS/IN/')

os.makedirs(IN + '/HDRI', exist_ok=True)
count = 0
for fname in os.listdir(SHARED + '/HDRI'):
    if fname.endswith('.hdr'):
        shutil.copy(SHARED + '/HDRI/' + fname, IN + '/HDRI/' + fname)
        print('✅ HDRI/' + fname + ' → F02_OCULUS/IN/HDRI/')
        count += 1

print('\n✅ Transit SHARED→F02 terminé — ' + str(count) + ' fichier(s) HDRI copié(s)')

## Étape 2 — Valider les fichiers IN/ (L'Arbitre CHECK-IN)

In [ ]:
import urllib.request, subprocess

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/kioka8877-ux/PALATIUM/main/PAL_ARBITRE.py',
    '/content/PAL_ARBITRE.py'
)
print('✅ Arbitre téléchargé')

result = get_ipython().getoutput(
    'python /content/PAL_ARBITRE.py --frigate F02 --mode check-in --drive-root ' + DRIVE_ROOT
)
print('\n'.join(result))

ret = subprocess.run(['python', '/content/PAL_ARBITRE.py',
    '--frigate', 'F02', '--mode', 'check-in', '--drive-root', DRIVE_ROOT],
    capture_output=True)
if ret.returncode != 0:
    raise RuntimeError('F02 BLOQUÉE — Transférer les fichiers IN/ avant de continuer')
print('✅ F02 IN/ validé — prêt au lancement')

## Étape 3 — Copier les scripts dans Colab

In [ ]:
import urllib.request

GITHUB_RAW = 'https://raw.githubusercontent.com/kioka8877-ux/PALATIUM/main/F02_OCULUS/CODEBASE'

for fname in ['pal_f02_flask.py', 'pal_f02_viewer.html']:
    urllib.request.urlretrieve(GITHUB_RAW + '/' + fname, '/content/' + fname)
    print('✅ ' + fname)

## Étape 4 — Démarrer Flask (tunnel Colab)

> Utilise le proxy natif de Colab — aucun compte requis, URL HTTPS automatique.

In [ ]:
import threading, time
from pal_f02_flask import init_config, app, CONFIG
from google.colab.output import eval_js

init_config(DRIVE_ROOT)
CONFIG['viewer_path'] = '/content/pal_f02_viewer.html'

t = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=FLASK_PORT, debug=False, use_reloader=False)
)
t.daemon = True
t.start()

time.sleep(2)
url = eval_js('google.colab.kernel.proxyPort(' + str(FLASK_PORT) + ')')
print('✅ VIEWER DISPONIBLE ICI :')
print(url)

## Étape 5 — Utiliser le viewer

1. Ouvrir l'URL affichée ci-dessus dans votre navigateur
2. Naviguer dans la scène 3D
3. Définir le spawn point (touche **S**)
4. Ajouter des waypoints (touche **W**)
5. Valider les tags dans le panneau de droite
6. **Ctrl+S** pour sauvegarder (`creative_config.json` + `tags_config.json`)

> L'URL change à chaque session Colab — c'est normal.

## Étape 6 — Valider les fichiers OUT/ (L'Arbitre CHECK-OUT)

In [ ]:
import subprocess
ret = subprocess.run(['python', '/content/PAL_ARBITRE.py',
    '--frigate', 'F02', '--mode', 'check-out',
    '--drive-root', DRIVE_ROOT, '--verbose'],
    capture_output=False)

if ret.returncode == 0:
    print('\n  ✅ F02 OCULUS SCELLÉE — Transférer les configs vers F03')
else:
    print('\n  ❌ Configs manquantes — Sauvegarder depuis le viewer (Ctrl+S)')

---
## Résumé Transit

Après CHECK-OUT ✅ :

| Fichier | Source | Destination |
|---------|--------|-------------|
| `creative_config.json` | `F02_OCULUS/OUT/` | `F03_SCRIPTORIUM/IN/` |
| `tags_config.json` | `F02_OCULUS/OUT/` | `F03_SCRIPTORIUM/IN/` |

**Inscrire dans `TRACKING/PALATIUM_TRANSFER_LOG.md`.**

*F02 OCULUS — Scellée. Ad Victoriam.*